# Figures for the paper

Every figure in `paper.md` is produced here, from the committed result files in `../results/*.json`.
Nothing is re-trained and no GPU is needed: this notebook is a plotting job.

**Two kinds of figure live here, and they are labelled:**

| kind | what it is | where it may appear |
|---|---|---|
| **measured** | every number is read from `../results/*.json`, produced by `src/` | anywhere |
| **conceptual** | a schematic whose geometry is invented to make an idea visible; annotated numbers are still measured | Introduction and Related work only |

Run: `python -m nbconvert --to notebook --execute --inplace figures.ipynb`

Palette: slate = representation-fitted bridges, blue = behaviour-fitted bridges, amber = cheap
repairs of the representation-fitted bridge, red dashed = oracle ceiling, teal = oracle/geometry.

In [1]:
from pathlib import Path
import json

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Ellipse, Rectangle
from matplotlib.lines import Line2D

RES = Path("../results")
FIG = Path("figures")
FIG.mkdir(exist_ok=True)

INK, MUT, GRID = "#111827", "#6B7280", "#E5E7EB"
BLUE, LBLUE, DBLUE = "#2563EB", "#93C5FD", "#1D4ED8"
RED, AMBER, LAMBER = "#DC2626", "#B45309", "#D97706"
TEAL, SLATE, LSD, PALE = "#0D9488", "#334155", "#94A3B8", "#E2E8F0"

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "figure.facecolor": "w", "axes.facecolor": "w",
    "font.size": 9, "axes.titlesize": 10.5, "axes.labelsize": 9,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "axes.edgecolor": INK, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7,
    "legend.frameon": False, "mathtext.fontset": "dejavusans",
    "figure.constrained_layout.use": True,
})


def load(name):
    with open(RES / f"{name}.json", encoding="utf-8") as f:
        return json.load(f)


CAL, COIN = load("calibration"), load("coincidence")
BOOT, COM = load("bootstrap"), load("comet")
REP, M48 = load("replicate"), load("maxnew48")
BAS, BM = load("basis_analysis"), load("behaviour_maps")


def save(fig, name):
    fig.savefig(FIG / f"{name}.png")
    print("wrote", FIG / f"{name}.png", tuple(round(v, 2) for v in fig.get_size_inches()))
    return fig


ARMS = {
    "ce_scratch":      dict(lab="CE from identity, 2 ep", bleu=2.245, chrf=17.28, ce=3.314,
                            degen=0.265, comet=0.4363, norm=24.4, zdist=33.7, g="neg", short="CE from identity"),
    "ridge_sub":       dict(lab="ridge, same 20k pairs", bleu=2.474, chrf=19.68, ce=3.232,
                            degen=0.167, comet=0.4751, norm=7.6, zdist=6.2, g="ridge", short="ridge (20k)"),
    "ridge_full":      dict(lab="ridge, published map", bleu=2.499, chrf=19.77, ce=3.228,
                            degen=0.155, comet=0.4797, norm=7.6, zdist=6.2, g="ridge", short="ridge (published)"),
    "ridge_scale122":  dict(lab="ridge $\\times$ 1.22", bleu=2.888, chrf=20.88, ce=3.248,
                            degen=0.190, comet=0.4898, norm=9.3, zdist=7.9, g="cheap", short="ridge x1.22"),
    "ridge_recolor10": dict(lab="ridge + recolour $t$=0.1", bleu=3.085, chrf=20.70, ce=3.262,
                            degen=0.144, comet=0.4966, norm=8.8, zdist=8.6, g="cheap", short="ridge + recolour"),
    "ce_ridge":        dict(lab="behaviour, 2 ep", bleu=5.332, chrf=25.00, ce=2.857,
                            degen=0.118, comet=0.5331, norm=9.3, zdist=10.6, g="ce", short="behaviour 2 ep"),
    "ce_ridge_e6":     dict(lab="behaviour, 6 ep", bleu=7.045, chrf=27.24, ce=2.761,
                            degen=0.090, comet=0.5641, norm=11.3, zdist=14.3, g="ce", short="behaviour 6 ep"),
    "distill":         dict(lab="behaviour + KL, 8 ep", bleu=7.254, chrf=27.57, ce=2.741,
                            degen=0.104, comet=0.5678, norm=11.9, zdist=15.4, g="kl", short="behaviour + KL"),
    "ce_ridge_e16":    dict(lab="behaviour, 16 ep", bleu=7.897, chrf=29.36, ce=2.738,
                            degen=0.067, comet=0.5830, norm=13.8, zdist=18.8, g="ce", short="behaviour 16 ep"),
}
ORDER = ["ce_scratch", "ridge_sub", "ridge_full", "ridge_scale122", "ridge_recolor10",
         "ce_ridge", "ce_ridge_e6", "distill", "ce_ridge_e16"]
COL = {"neg": PALE, "ridge": SLATE, "cheap": LAMBER, "ce": BLUE, "kl": TEAL}

ORACLE = CAL["channels"]["ref"]
NATIVE = CAL["channels"]["native"]
assert abs(ORACLE["bleu"] - 74.22) < 0.05, ORACLE

## 1. The hook: meaning has structure before it has words

**Conceptual.** No measured geometry: the point is that two models can embed the same relational
structure in different coordinate systems, so a map between the systems is the only missing object.

In [2]:
rng = np.random.default_rng(11)
names = ["apple", "run", "red", "more", "water", "I", "house"]
layout = np.array([[-1.7, 0.3], [-0.3, -1.6], [1.0, 1.3], [1.9, 0.0],
                   [1.0, -1.3], [-1.0, -0.7], [-0.1, 1.6]])
edges = [(0, 2), (0, 1), (2, 3), (5, 0), (4, 1), (4, 6), (6, 2), (1, 3)]

th = -0.62
R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
other = (layout * np.array([1.05, 0.72])) @ R.T * 1.15 + np.array([0.5, -0.15])

fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.9))

# (a) shared relations
ax = axes[0]
for i, j in edges:
    ax.plot(*zip(layout[i], layout[j]), color=LSD, lw=1.0, zorder=1)
ax.scatter(*layout.T, s=95, color=SLATE, zorder=3, edgecolor="w", linewidth=1.2)
offs = [(-11, 8, "right"), (10, -10, "left"), (10, 3, "left"), (10, -3, "left"),
        (10, -5, "left"), (-10, -9, "right"), (7, 8, "left")]
for (x, y), n, (dx, dy, ha) in zip(layout, names, offs):
    ax.annotate(n, (x, y), xytext=(dx, dy), textcoords="offset points",
                fontsize=9.2, color=INK, ha=ha, va="center")
ax.set_title("(a)  the relations", loc="left")
ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.3, 2.3)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.text(0.0, -2.12, "what a sentence is about", ha="center", fontsize=8.2, color=MUT)

# (b) and (c): the same graph in two coordinate systems
for ax, pts, col, title, note in [(axes[1], layout, SLATE, "(b)  Vietnamese model", "its 512 numbers"),
                                  (axes[2], other, TEAL, "(c)  English model", "the same 512 numbers, other units")]:
    for i, j in edges:
        ax.plot(*zip(pts[i], pts[j]), color=col, lw=1.0, alpha=0.6, zorder=1)
    ax.scatter(*pts.T, s=80, color=col, zorder=3, edgecolor="w", linewidth=1.1)
    ax.set_title(title, loc="left")
    ax.set_xlim(-2.9, 2.9); ax.set_ylim(-2.3, 2.3)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.text(0.0, -2.12, note, ha="center", fontsize=8.2, color=MUT)

fig.suptitle("Translation as coordinate change, not as learning a language", fontsize=11.5, x=0.02, ha="left")
save(fig, "fig01_concept_space")

wrote figures\fig01_concept_space.png (np.float64(11.0), np.float64(3.9))


<Figure size 1210x429 with 3 Axes>

## 2. Four ways to connect two frozen models

**Conceptual** wiring; the BLEU values printed under each panel are **measured** on the same 1,000
held-out sentences. `0.05` is the naive no-bridge control (identity channel, same adapter);
`74.2` is the oracle described in §5.

In [3]:
def box(ax, x, y, w, h, title, sub="", fc="#F8FAFC", ec=INK, ls="-", lw=1.0, tc=INK, fs=8.6):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.10",
                                fc=fc, ec=ec, lw=lw, ls=ls, zorder=2))
    ax.text(x + w / 2, y + h * (0.63 if sub else 0.5), title, ha="center", va="center",
            fontsize=fs, color=tc, zorder=3)
    if sub:
        ax.text(x + w / 2, y + h * 0.26, sub, ha="center", va="center", fontsize=fs - 1.4,
                color=MUT, zorder=3)


def arrow(ax, p, q, ls="-", color=INK, lw=1.15, rad=0.0):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=11, lw=lw, color=color,
                                 ls=ls, zorder=4, shrinkA=1, shrinkB=1,
                                 connectionstyle=f"arc3,rad={rad}"))


fig, axes = plt.subplots(2, 2, figsize=(10.0, 4.9))
panels = [
    ("(a)  No bridge", "the naive form of the hypothesis", 0.05,
     "the decoder receives source-language states\nwith nothing in between"),
    ("(b)  A mapped bridge", "one 263k-parameter matrix", 7.90,
     "this paper: the only difference from (a)"),
    ("(c)  Target encoder on the source", "the decoder's own encoder, reading Vietnamese", 3.75,
     "no cross-lingual transfer at all"),
    ("(d)  Oracle", "target encoder reading the English reference", 74.22,
     "not available at test time"),
]
for ax, (title, sub, bleu, note) in zip(axes.ravel(), panels):
    ax.set_xlim(0, 10); ax.set_ylim(0, 5.2)
    ax.axis("off")
    ax.set_title(title, loc="left", fontsize=10)
    ax.text(0, 4.72, sub, fontsize=8.2, color=MUT, ha="left")

    yb, hb = 2.55, 1.05
    if title.startswith("(a)"):
        box(ax, 0.3, yb, 2.6, hb, r"$M_{vi}$ encoder", "frozen", fc="#F1F5F9")
        box(ax, 6.9, yb, 2.8, hb, r"$M_{en}$ decoder", "+ rank-16 LoRA", fc="#F1F5F9")
        arrow(ax, (2.9, yb + hb / 2), (6.9, yb + hb / 2), color=MUT)
        ax.text(4.9, yb + hb / 2 + 0.24, "no map", fontsize=8.2, color=MUT, ha="center")
    if title.startswith("(b)"):
        box(ax, 0.3, yb, 2.2, hb, r"$M_{vi}$ encoder", "frozen", fc="#F1F5F9")
        box(ax, 3.1, yb - 0.06, 2.5, hb + 0.12, r"$f(h)=Wh+b$", "263k, fitted", fc="#DBEAFE", ec=BLUE)
        box(ax, 6.6, yb, 3.1, hb, r"$M_{en}$ decoder", "+ rank-16 LoRA", fc="#F1F5F9")
        arrow(ax, (2.5, yb + hb / 2), (3.1, yb + hb / 2), color=BLUE)
        arrow(ax, (5.6, yb + hb / 2), (6.6, yb + hb / 2), color=BLUE)
        ax.text(3.1, yb + hb + 0.22, "trained here", fontsize=8, color=BLUE, ha="left")
    if title.startswith("(c)"):
        box(ax, 0.3, yb, 3.0, hb, r"$M_{en}$ encoder", "reads $s_{vi}$", fc="#F1F5F9")
        box(ax, 6.9, yb, 2.8, hb, r"$M_{en}$ decoder", "+ rank-16 LoRA", fc="#F1F5F9")
        arrow(ax, (3.3, yb + hb / 2), (6.9, yb + hb / 2), color=SLATE)
        ax.text(5.1, yb + hb / 2 + 0.24, r"$E_{en}(s_{vi})$", fontsize=8.2, ha="center", color=SLATE)
    if title.startswith("(d)"):
        box(ax, 0.3, yb, 3.0, hb, r"$M_{en}$ encoder", "reads $s_{en}$", fc="#CCFBF1", ec=TEAL)
        box(ax, 6.9, yb, 2.8, hb, r"$M_{en}$ decoder", "+ rank-16 LoRA", fc="#F1F5F9")
        arrow(ax, (3.3, yb + hb / 2), (6.9, yb + hb / 2), color=TEAL, ls=(0, (4, 2)))
        ax.text(5.1, yb + hb / 2 + 0.24, r"$H_{ref}$", fontsize=8.2, ha="center", color=TEAL)

    ax.text(0.3, 0.75, f"{bleu:.2f} BLEU", fontsize=14, color=INK, ha="left", va="center")
    ax.text(0.3, 0.18, note, fontsize=8.0, color=MUT, ha="left", va="center")

fig.suptitle("Same decoder, four memories: what a bridge has to supply", fontsize=11.5, x=0.02, ha="left")
save(fig, "fig02_channels")

wrote figures\fig02_channels.png (np.float64(10.0), np.float64(4.9))


<Figure size 1100x539 with 4 Axes>

## 3. Where the literature sits, and where the gap is

**Conceptual** map of the works cited in §2. Positions are the authors' reading, not a measurement:
the horizontal axis is *how much of the system is trained*, the vertical axis is *how close the
evaluation gets to generated text*. Bubble size is not data.

In [4]:
# ---- hand-set coordinates: (trained, evaluated, label, dx, dy, ha, cluster)
P = [
    # x = what is trained: 0 nothing | 1 bridge | 2 readout | 3 generator
    # y = what is evaluated: 0 representation | 1 task accuracy | 2 generated text
    (0.55, 0.12, "convergence of\nmonolingual spaces", 0, -20, "center", "rep"),
    (1.05, 0.32, "embedding alignment\n(ridge, Procrustes)", 12, 0, "left", "rep"),
    (1.30, 0.68, "model stitching", 12, 0, "left", "rep"),
    (0.35, 0.98, "latent-space\ncommunication", 10, 0, "left", "rep"),
    (2.55, 0.60, "middle-layer /\nlayer-wise alignment", -8, 10, "right", "train"),
    (3.00, 1.05, "alignment inside a\ntrained MT model", -8, -10, "right", "train"),
    (2.30, 1.45, "external translator\nin the loop", -10, 0, "right", "train"),
    (1.50, 2.05, "this paper", 12, 0, "left", "us"),
]
CLUSTER = dict(rep=dict(c=SLATE, fc="#F1F5F9"), train=dict(c=AMBER, fc="#FEF3C7"),
               us=dict(c=BLUE, fc="#DBEAFE"))

fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.grid(False)
ax.axvspan(-0.2, 1.85, color="#F8FAFC", zorder=0)
ax.axvspan(1.85, 3.45, color="#FFFBEB", zorder=0)
ax.text(0.82, -0.62, "the models stay frozen", ha="center", fontsize=8.2, color=MUT)
ax.text(2.65, -0.62, "the models are touched", ha="center", fontsize=8.2, color=MUT)

for x, y, lab, dx, dy, ha, cl in P:
    s = CLUSTER[cl]
    ax.scatter(x, y, s=170 if cl == "us" else 85, color=s["c"], zorder=3,
               edgecolor="w", linewidth=1.2)
    ax.annotate(lab, (x, y), xytext=(dx, dy), textcoords="offset points",
                fontsize=8.3, color=INK if cl == "us" else MUT, ha=ha, va="center",
                fontweight="bold" if cl == "us" else "normal")

ax.annotate("", xy=(1.44, 1.92), xytext=(1.12, 0.48),
            arrowprops=dict(arrowstyle="-|>", lw=1.4, color=BLUE, ls=(0, (4, 2)),
                            connectionstyle="arc3,rad=-0.45"))
ax.text(0.72, 1.44, "the missing measurement:\nfit on representation, judge on text",
        fontsize=8.4, color=BLUE, ha="center")

ax.set_xlim(-0.2, 3.45); ax.set_ylim(-0.8, 2.45)
ax.set_xlabel("how much of the system is trained  $\\longrightarrow$")
ax.set_ylabel("how close the evaluation is to generated text  $\\longrightarrow$")
ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(["nothing", "bridge", "readout", "generator"])
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["representation\nmetric", "task\naccuracy", "generated\ntext"])
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.set_title("Nobody evaluates generation from a bridge fitted to be read", loc="left")
save(fig, "fig03_literature_map")

wrote figures\fig03_literature_map.png (np.float64(8.4), np.float64(4.6))


<Figure size 924x506 with 1 Axes>

## 4. The object the paper is about: the decoder's tolerance shell

**Conceptual** geometry, **measured** widths. The two axes are two directions of memory space, chosen
to be one the decoder reads and one it does not. The band's width comes from measurement: at
$\varepsilon=0.05$ in a random direction 78% of outputs are unchanged, while along the gradient
$\varepsilon=0.05$ changes all of them. The least-squares bridge is placed at the measured
displacement ($\varepsilon=1$ along its own error direction).

In [5]:
fig, ax = plt.subplots(figsize=(8.8, 4.0))
ax.grid(False)

w = 0.13                                   # half-width of the tolerance band
u = np.linspace(-1.35, 1.4, 400)
centre = 0.10 + 0.16 * u + 0.09 * np.sin(2.0 * u)

ax.fill_between(u, centre - w, centre + w, color=TEAL, alpha=0.30, lw=0, zorder=1)
ax.plot(u, centre - w, color=TEAL, lw=1.4, zorder=2)
ax.plot(u, centre + w, color=TEAL, lw=1.4, zorder=2)
ax.plot(u, centre, color="#0F766E", lw=0.8, ls=(0, (3, 3)), alpha=0.9, zorder=3)
for k, alpha in [(2.2, 0.55), (4.2, 0.30), (7.0, 0.16)]:
    ax.plot(u, centre - k * w, color="#9CA3AF", lw=0.9, alpha=alpha, zorder=1)
    ax.plot(u, centre + k * w, color="#9CA3AF", lw=0.9, alpha=alpha, zorder=1)

ax.scatter([0], [0.10], s=70, color=INK, zorder=6, edgecolor="w", linewidth=1.0)
ax.annotate("$H^{*}$: the centre of the band — a memory\nthe decoder reads perfectly (74 BLEU)",
            (0, 0.03), xytext=(0.18, -0.36), fontsize=8.4, color=INK,
            arrowprops=dict(arrowstyle="-|>", lw=0.8, color=INK))

# the band width
xs = -1.15
ax.annotate("", xy=(xs, centre[0] - w - 0.003), xytext=(xs, centre[0] + w + 0.003),
            arrowprops=dict(arrowstyle="<|-|>", lw=1.0, color="#0F766E"), zorder=5)
ax.text(xs, centre[0] + w + 0.10, "width $\\approx 0.2\\,\\|H\\|$", fontsize=8.2,
        color="#0F766E", ha="center", va="bottom")

# the least-squares bridge
xr, yr = 0.95, 0.80
ax.scatter([xr], [yr], s=75, color=RED, zorder=6)
th = np.linspace(0, 2 * np.pi, 200)
ax.plot(xr + w * np.cos(th), yr + w * np.sin(th), color=RED, lw=1.0, ls=":", zorder=5)
ax.annotate("the least-squares bridge\n2.5 BLEU, 0% of outputs identical",
            (xr, yr), xytext=(xr + 0.16, yr + 0.06), fontsize=8.4, color=RED, va="center")
ax.annotate("", xy=(xr, 0.10 + 0.16 * xr + 0.09 * np.sin(2.0 * xr)), xytext=(xr, yr - 0.04),
            arrowprops=dict(arrowstyle="-|>", lw=1.4, color=RED, shrinkA=6, shrinkB=2))
ax.text(xr, yr - w - 0.10, "its error is isotropic", fontsize=8.0, color=RED,
        ha="center", va="top")

ax.set_xlim(-1.4, 1.9); ax.set_ylim(-0.55, 1.10)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel("memory coordinate 1  $\\longrightarrow$")
ax.set_ylabel("coordinate 2")
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("The decoder is not a metric ball: its tolerance is a thin band", loc="left")
save(fig, "fig04_level_set")

wrote figures\fig04_level_set.png (np.float64(8.8), np.float64(4.0))


<Figure size 968x440 with 1 Axes>

## 5. Method overview

**Conceptual** diagram of the pipeline, with tensor shapes. The only difference between the two
objectives is the loss attached to the same 263k-parameter map.

In [6]:
fig, ax = plt.subplots(figsize=(9.6, 4.9))
ax.set_xlim(0, 10.6); ax.set_ylim(0, 6.2); ax.axis("off")

# ---- row 1: the channel
box(ax, 0.15, 4.25, 1.45, 0.9, "Vietnamese", "sentence $s_{vi}$", fc="#F1F5F9")
box(ax, 2.0, 4.25, 1.45, 0.9, "encoder $E_1$", "frozen", fc="#F1F5F9")
box(ax, 3.85, 4.25, 1.55, 0.9, "memory $H_{vi}$", "$64\\times 512$", fc="#F8FAFC")
box(ax, 5.75, 4.20, 1.75, 1.0, "$f_\\theta(h)=Wh+b$", "263k, trainable", fc="#DBEAFE", ec=BLUE)
box(ax, 7.9, 4.25, 1.55, 0.9, "decoder $D_2$", "+ LoRA, frozen", fc="#F1F5F9")
for x0, x1 in [(1.6, 2.0), (3.45, 3.85), (5.4, 5.75), (7.5, 7.9)]:
    arrow(ax, (x0, 4.7), (x1, 4.7), color=INK if x1 < 5.9 else BLUE)
ax.text(9.55, 4.7, "$\\rightarrow$ English", fontsize=8.4, color=MUT, va="center")

# ---- row 2: the reference side
box(ax, 0.15, 2.45, 1.45, 0.9, "English", "sentence $s_{en}$", fc="#F1F5F9")
box(ax, 2.0, 2.45, 1.45, 0.9, "encoder $E_2$", "frozen", fc="#F1F5F9")
box(ax, 3.85, 2.45, 1.55, 0.9, "memory $H_{en}$", "$T_{en}\\times 512$", fc="#F8FAFC")
box(ax, 5.75, 2.45, 1.75, 0.9, "alignment", "character-span pooling", fc="#F8FAFC")
box(ax, 7.9, 2.45, 1.55, 0.9, "target $\\tilde{H}_{en}$", "at $s_{vi}$ positions", fc="#F8FAFC")
for x0, x1 in [(1.6, 2.0), (3.45, 3.85), (5.4, 5.75), (7.5, 7.9)]:
    arrow(ax, (x0, 2.9), (x1, 2.9), color=INK)

# ---- row 3: the two losses
box(ax, 3.55, 0.30, 3.65, 0.95, "$\\mathcal{L}_{emb}=\\|f(H_{vi})-\\tilde{H}_{en}\\|^2$",
    "closed form on $f(H_{vi})$ and $\\tilde{H}_{en}$", fc=PALE, ec=MUT, fs=8.0)
box(ax, 7.75, 0.30, 2.75, 0.95, "$\\mathcal{L}_{beh}=-\\log p(s_{en}\\mid f(H_{vi}))$",
    "gradient to $f$ through $D_2$", fc="#DBEAFE", ec=BLUE, fs=8.0)
ax.text(3.55, 1.42, "what the literature fits", fontsize=8.4, color=MUT, ha="left")
ax.text(7.75, 1.42, "what the decoder is judged by", fontsize=8.4, color=BLUE, ha="left")

ax.text(0.15, 5.95, "One map, two possible losses", fontsize=11.5, ha="left")
save(fig, "fig05_pipeline")

wrote figures\fig05_pipeline.png (np.float64(9.6), np.float64(4.9))


<Figure size 1056x539 with 1 Axes>

## 6. Main result: BLEU, full range and zoom

**Measured.** n = 1,000 held-out sentences, greedy, `max_new=64`. The oracle (red dashed) is the
same decoder reading the target encoder's states of the English reference: a behavioural ceiling,
not a competitor.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.0))
y = np.arange(len(ORDER))[::-1]
vals = [ARMS[a]["bleu"] for a in ORDER]

for ax, xmax, title in [(axes[0], 80, "(a)  against the readout's ceiling"),
                        (axes[1], 9.6, "(b)  zoom")]:
    ax.barh(y, vals, height=0.62, color=[COL[ARMS[a]["g"]] for a in ORDER], zorder=3)
    ax.set_yticks(y)
    ax.set_xlim(0, xmax)
    ax.grid(axis="y", visible=False)
    ax.set_title(title, loc="left")
    ax.set_xlabel("BLEU")
    for yy, v in zip(y, vals):
        ax.text(v + xmax * 0.012, yy, f"{v:.1f}", va="center", fontsize=7.6, color=INK)

axes[0].set_yticklabels([ARMS[a]["lab"] for a in ORDER])
axes[0].tick_params(axis="y", length=0)
axes[1].set_yticklabels([])
axes[0].set_ylim(-0.7, 9.1)
axes[1].set_ylim(-0.7, 9.1)

axes[0].axvline(ORACLE["bleu"], color=RED, lw=1.1, ls=(0, (5, 3)), alpha=0.85, zorder=2)
axes[0].text(ORACLE["bleu"] - 1.5, 8.35, "oracle ceiling  74.2 BLEU", color=RED, fontsize=8.2,
             ha="right", va="center")
axes[0].axvspan(7.9, ORACLE["bleu"], color=RED, alpha=0.05, lw=0, zorder=0)
axes[0].text(30, 3.0, "89% of the ceiling\nunexplored", fontsize=9.2, color=RED, ha="center")

fig.suptitle("Fitting the bridge on behaviour multiplies BLEU by 3.2 — and leaves 89% of the ceiling on the table",
             fontsize=11.2, x=0.02, ha="left")
save(fig, "fig06_main_bleu")

wrote figures\fig06_main_bleu.png (np.float64(10.6), np.float64(4.0))


<Figure size 1166x440 with 2 Axes>

## 7. The same arms on the other instruments

**Measured.** chrF and reference cross-entropy are per-token/per-character and therefore not
diluted by the noisy references; COMET is an independent learned metric on 400 sentences.

In [8]:
fig, axes = plt.subplots(1, 4, figsize=(12.2, 3.9), sharey=True)
metrics = [("chrf", "chrF", 0, 90, "higher better", ORACLE["chrf"], "oracle 84.7"),
           ("ce", "reference CE (nats/token)", 2.4, 3.45, "lower better", None,
            "$\\leftarrow$ oracle 0.30, off scale"),
           ("degen", "degenerate outputs (fraction)", 0, 0.30, "lower better", ORACLE["degen"], "oracle 0.01"),
           ("comet", "COMET", 0.42, 0.60, "higher better", None, "n = 400")]

for ax, (key, lab, lo, hi, note, ref, ref_lab) in zip(axes, metrics):
    vals = [ARMS[a][key] for a in ORDER]
    ax.barh(y, vals, height=0.62, color=[COL[ARMS[a]["g"]] for a in ORDER], zorder=3)
    ax.set_xlim(lo, hi)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel(lab)
    ax.set_title(note, loc="left", fontsize=8.6, color=MUT)
    span = hi - lo
    for yy, v in zip(y, vals):
        fmt = f"{v:.2f}" if key == "comet" else (f"{v:.0%}" if key == "degen" else f"{v:.1f}")
        ax.text(v + span * 0.012, yy, fmt, va="center", ha="left", fontsize=7.4, color=INK)

    if ref is not None:
        ax.axvline(ref, color=RED, lw=1.0, ls=(0, (5, 3)), alpha=0.75, zorder=2)
        ax.text(ref, 8.75, ref_lab, color=RED, fontsize=7.6,
                ha="left" if ref < lo + span * 0.5 else "right", va="center")
    else:
        ax.text(lo + span * 0.03, 8.75, ref_lab, fontsize=7.6, color=MUT, va="center")
axes[0].set_yticks(y)
axes[0].set_yticklabels([ARMS[a]["lab"] for a in ORDER])
axes[0].tick_params(axis="y", length=0)
axes[0].set_ylim(-0.7, 9.3)
fig.suptitle("Four instruments, one ordering: the behaviour-fitted arms win on everything except absolute quality",
             fontsize=11.2, x=0.02, ha="left")
save(fig, "fig07_internal_metrics")

wrote figures\fig07_internal_metrics.png (np.float64(12.2), np.float64(3.9))


<Figure size 1342x429 with 4 Axes>

## 8. The decoder's tolerance, measured

**Measured.** n = 300 sentences. `eps` is displacement as a fraction of the oracle memory's own
Frobenius norm. Left: random direction, i.e. the isotropic null hypothesis. Middle: along the
direction that separates the ridge memory from the oracle memory (`eps = 1` *is* the ridge
memory). Right: the same displacement magnitude in a random direction and along the steepest
direction of the decoder's own loss.

In [9]:
ref = COIN["oracles"]["ref"]
rand = ref["curves"]["random"]
ridg = ref["curves"]["ridge"]
nat = COIN["oracles"]["nat"]

fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.7))

ax = axes[0]
e = [c["eps"] for c in rand]
ax.plot(e, [c["bleu"] for c in rand], "o-", color=TEAL, lw=1.6, ms=4, label="BLEU")
ax.plot(e, [100 * c["exact"] for c in rand], "s--", color=SLATE, lw=1.2, ms=3.6,
        label="outputs identical (%)")
ax.set_xlabel("displacement $\\varepsilon$ (fraction of $\\|H^{*}\\|$)")
ax.set_ylabel("% / BLEU")
ax.set_title("(a)  a random nudge barely matters", loc="left")
ax.annotate("5% of the norm\nalready changes 22%\nof the sentences", (0.05, 78), xytext=(0.22, 47),
            fontsize=8.2, color=INK, arrowprops=dict(arrowstyle="-|>", lw=0.8, color=MUT))
ax.legend(loc="center right", fontsize=8)
ax.set_xlim(0, 0.86)
ax.set_ylim(0, 100)

ax = axes[1]
e = [c["eps"] for c in ridg]
ax.plot(e, [c["bleu"] for c in ridg], "o-", color=RED, lw=1.6, ms=4)
ax.axvline(1.0, color=MUT, lw=1.0, ls=(0, (4, 3)))
ax.text(1.02, 63, "the least-squares\nbridge sits here", fontsize=8.2, color=MUT, va="top")
ax.annotate("74 BLEU", (0.25, 70.4), xytext=(0.36, 74), fontsize=8.2, color=RED,
            arrowprops=dict(arrowstyle="-", lw=0.7, color=RED))
ax.annotate("2.4 BLEU", (1.0, 2.36), xytext=(1.08, 12), fontsize=8.4, color=RED)
ax.set_xlabel("displacement along the bridge's own error")
ax.set_ylabel("BLEU")
ax.set_title("(b)  the bridge's error is not slight", loc="left")
ax.set_xlim(0.2, 1.55); ax.set_ylim(0, 80)

ax = axes[2]
labels = ["random", "steepest\ndirection"]
ce = [nat["curves"]["random"][0]["ce"], nat["curves"]["grad"][0]["ce"]]
bleu = [nat["curves"]["random"][0]["bleu"], nat["curves"]["grad"][0]["bleu"]]
xx = np.arange(2)
ax.bar(xx - 0.19, ce, width=0.34, color=[LSD, RED], zorder=3, label="CE (nats)")
ax.set_ylabel("reference CE")
ax.set_ylim(0, 8.2)
for i, v in enumerate(ce):
    ax.text(i - 0.19, v + 0.12, f"{v:.2f}", ha="center", fontsize=8.2)
ax2 = ax.twinx()
ax2.bar(xx + 0.19, bleu, width=0.34, color=[PALE, "#FEE2E2"], edgecolor=[LSD, RED], zorder=3,
        label="BLEU")
ax2.set_ylabel("BLEU")
ax2.set_ylim(0, 8.2)
ax2.grid(False)
for i, v in enumerate(bleu):
    ax2.text(i + 0.19, v + 0.12, f"{v:.1f}", ha="center", fontsize=8.2, color=RED)
ax.set_xticks(xx); ax.set_xticklabels(labels)
ax.set_title("(c)  same $\\varepsilon=0.05$, 2.1$\\times$ the loss", loc="left")
ax.text(0.5, 7.4, "the decoder is anisotropic\nby at least two orders", ha="center", fontsize=8.2,
        color=RED)

fig.suptitle("Why a small vector error is a large behavioural error", fontsize=11.2, x=0.02, ha="left")
save(fig, "fig08_tolerance")

wrote figures\fig08_tolerance.png (np.float64(11.6), np.float64(3.7))


<Figure size 1276x407 with 4 Axes>

## 9. Where the least-squares error lives

**Measured** on 1,000 sentences, per-sentence error energy normalised per sentence. The subspace is
the top-$k$ right singular directions of the decoder's per-sentence gradient $\\partial\\mathrm{CE}/\\partial H$
at the oracle memory. If the error hid in a null space, the bars would be near zero.

In [10]:
sub = COIN["oracles"]["nat"]["sensitive_subspace"]["energy_in_topk"]
ks = [1, 8, 64, 256]
obs = [sub[str(k)]["energy_frac"] for k in ks]
ch = [sub[str(k)]["chance"] for k in ks]
enr = [o / c for o, c in zip(obs, ch)]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))

ax = axes[0]
ax.loglog(ks, ch, "o--", color=LSD, lw=1.2, ms=4, label="chance (isotropic error)")
ax.loglog(ks, obs, "o-", color=SLATE, lw=1.6, ms=5, label="ridge error")
ax.set_xlabel("size $k$ of the decoder's most sensitive subspace")
ax.set_ylabel("share of the error's energy")
ax.set_title("(a)  the error is spread almost evenly", loc="left")
ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.minorticks_off()
ax.legend(fontsize=8, loc="lower right")
ax.text(1.15, 6.5e-3, "even at $k=256$ the read subspace\nholds 1% of the error", fontsize=8,
        color=MUT, ha="left", va="top")

ax = axes[1]
ax.bar(range(len(ks)), enr, color=[PALE if e < 1 else "#CBD5E1" for e in enr], width=0.6, zorder=3)
ax.axhline(1.0, color=MUT, lw=1.0, ls=(0, (4, 3)))
ax.text(-0.42, 1.06, "chance", fontsize=8, color=MUT, ha="left")
ax.set_xticks(range(len(ks))); ax.set_xticklabels([f"top-{k}" for k in ks])
ax.set_ylabel("observed / chance")
ax.set_ylim(0, 2.2)
for i, e in enumerate(enr):
    ax.text(i, e + 0.06, f"{e:.1f}$\\times$", ha="center", fontsize=8.6)
ax.set_title("(b)  a null-space rescue would need $\\approx 0\\times$", loc="left")
ax.text(1.5, 1.95, "mild concentration, far from\nenough to hide the error", fontsize=8.2,
        color=MUT, ha="center")

fig.suptitle("The least-squares residual is direction-blind", fontsize=11.2, x=0.02, ha="left")
save(fig, "fig09_error_geometry")

wrote figures\fig09_error_geometry.png (np.float64(9.6), np.float64(3.6))


<Figure size 1056x396 with 2 Axes>

## 10. Controls: what the gain is *not*

**Measured.** Left: paired bootstrap over 1,000 sentences, 200 resamples, versus the published
ridge map; the band is the 95% interval. Right: the same cheap repairs applied to the same map,
as a share of the 2-epoch gain (+2.83 BLEU). Shares are descriptive, not additive.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.9), gridspec_kw=dict(width_ratios=[1.35, 1.0]))

arm_list = ["ridge_sub", "ce_scratch", "ridge_scale122", "ridge_recolor10",
            "ce_ridge", "ce_ridge_e6", "distill", "ce_ridge_e16"]
yy = np.arange(len(arm_list))[::-1]
ax = axes[0]
for i, a in zip(yy, arm_list):
    d = BOOT[a]["bleu"]
    c = DBLUE if ARMS[a]["g"] in ("ce", "kl") else (LAMBER if ARMS[a]["g"] == "cheap" else SLATE)
    ax.plot(d["ci"], [i, i], color=c, lw=2.2, solid_capstyle="round", zorder=3)
    ax.plot(d["delta"], i, "o", color=c, ms=6, zorder=4)
    ax.text(d["ci"][1] + 0.12, i, f"{d['delta']:+.2f}", va="center", fontsize=8)
ax.axvline(0, color=INK, lw=1.0)
ax.set_yticks(yy); ax.set_yticklabels([ARMS[a]["lab"] for a in arm_list])
ax.tick_params(axis="y", length=0)
ax.set_xlabel("$\\Delta$BLEU vs the published map  [95% CI]")
ax.set_title("(a)  the effect is 5$\\times$ outside the ambiguity", loc="left")
ax.set_xlim(-1.0, 7.0)
ax.grid(axis="y", visible=False)

ax = axes[1]
parts = [("more data\n(same 20k pairs)", 0.00, PALE),
         ("isotropic scale\n$\\times$1.22", 0.39, LAMBER),
         ("covariance\n(recolour $t$=0.1)", 0.59, "#FCD34D"),
         ("directional, behaviour-only", 2.83 - 0.39 - 0.59, BLUE)]
ypos = np.arange(4)[::-1]
for yy, (lab, v, c) in zip(ypos, parts):
    ax.barh(yy, v, color=c, height=0.62, zorder=3)
    ax.text(-0.06, yy, lab, ha="right", va="center", fontsize=8.0, color=INK)
    ax.text(v + 0.05 if v < 1.5 else v - 0.05, yy, f"{v:.2f}", va="center",
            ha="left" if v < 1.5 else "right", fontsize=8.4,
            color=INK if v < 1.5 else "w")
ax.set_yticks([]); ax.set_ylim(-0.7, 3.6); ax.set_xlim(-1.35, 2.6)
ax.set_xlabel("BLEU, 2-epoch gain (+2.83)")
ax.set_title("(b)  cheap explanations cover a third", loc="left")
ax.grid(axis="y", visible=False)
ax.axvline(0, color=INK, lw=0.8)

fig.suptitle("The objective did the work: not the corpus, not the scale, not the covariance",
             fontsize=11.2, x=0.02, ha="left")
save(fig, "fig10_decomposition")

wrote figures\fig10_decomposition.png (np.float64(11.0), np.float64(3.9))


<Figure size 1210x429 with 2 Axes>

## 11. Training dynamics: the loss plateaus, the text keeps improving

**Measured.** The 16-epoch trajectory is the concatenation of three warm-started runs
(2 + 4 + 10 epochs); the model is the same at every boundary, so the curve is one run's history.
Right: the KL term in the distillation arm against the plain CE run at the same number of epochs.

In [12]:
hist = ([h["val_ce"] for h in BM["arms"]["ce_ridge"]["hist"]] +
        [h["val_ce"] for h in BM["arms"]["ce_ridge_e6"]["hist"]] +
        [h["val_ce"] for h in BM["arms"]["ce_ridge_e16"]["hist"]])
assert len(hist) == 16, len(hist)

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.8), gridspec_kw=dict(width_ratios=[1.5, 1.0]))

ax = axes[0]
ep = np.arange(2, 2 + len(hist))
ax.plot(ep, hist, "o-", color=BLUE, lw=1.6, ms=3.6, label="validation CE")
ax.set_xlabel("epoch"); ax.set_ylabel("validation CE (nats/token)", color=BLUE)
ax.tick_params(axis="y", colors=BLUE)
ax.set_ylim(2.78, 3.05)
ax.set_title("(a)  CE is flat from epoch 4; BLEU is not", loc="left")

ax2 = ax.twinx()
pts = [(2, 5.332), (6, 7.045), (16, 7.897)]
ax2.plot(*zip(*pts), "s-", color=RED, lw=1.6, ms=6, label="BLEU")
for x, v in pts:
    ax2.annotate(f"{v:.1f}", (x, v), xytext=(0, 9), textcoords="offset points",
                 fontsize=8.2, color=RED, ha="center")
ax2.set_ylabel("BLEU", color=RED); ax2.set_ylim(4, 9.4)
ax2.tick_params(axis="y", colors=RED); ax2.grid(False)

ax = axes[1]
runs = [("CE only", 2.8089, PALE), ("CE + KL to\nthe oracle", 2.8090, TEAL)]
ax.bar([0, 1], [r[1] for r in runs], color=[r[2] for r in runs], width=0.55, zorder=3)
for i, (_, v, _) in enumerate(runs):
    ax.text(i, v + 0.0004, f"{v:.4f}", ha="center", fontsize=8.6)
ax.set_ylim(2.800, 2.812); ax.set_xticks([0, 1]); ax.set_xticklabels([r[0] for r in runs])
ax.set_ylabel("validation CE after 2 extra epochs")
ax.set_title("(b)  distillation adds 0.0001 nats", loc="left")
ax.annotate("the student's memory cannot\nrepresent a CE = 0.30 distribution",
            (0.5, 2.8105), ha="center", fontsize=8, color=MUT)

fig.suptitle("A second-order version of the same lesson: the surrogate saturates before the task does",
             fontsize=11.2, x=0.02, ha="left")
save(fig, "fig11_training")

wrote figures\fig11_training.png (np.float64(10.4), np.float64(3.8))


<Figure size 1144x418 with 3 Axes>

## 12. Replication and metric independence

**Measured.** Left: the same fitted maps on a disjoint 1,000-sentence sample (rows 1000–1999) that
was never used in this package. Right: COMET against BLEU for every arm; COMET is a learned metric
that shares no term with BLEU.

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.9), gridspec_kw=dict(width_ratios=[1.1, 1.0]))

ax = axes[0]
slope = [("ridge_full", SLATE), ("ridge_recolor10", LAMBER),
         ("ce_ridge_e6", BLUE), ("ce_ridge_e16", DBLUE)]
for i, (a, c) in enumerate(slope):
    v0, v1 = ARMS[a]["bleu"], REP["arms"][a]["bleu"]
    ax.plot([0, 1], [v0, v1], "o-", color=c, lw=1.6, ms=5)
    ax.text(1.06, v1, f"{v1:.1f}", ha="left", va="center", fontsize=8, color=c)
    ax.text(-0.10, v0, ARMS[a]["short"], ha="right", va="center", fontsize=7.6, color=c)
ax.set_xlim(-0.95, 1.35); ax.set_xticks([0, 1])
ax.set_xticklabels(["test rows\n0–999", "test rows\n1000–1999"])
ax.set_ylabel("BLEU"); ax.set_ylim(0, 9.6)
ax.set_title("(a)  the gap reproduces on unseen sentences", loc="left")
ax.grid(axis="x", visible=False)

ax = axes[1]
xs = [ARMS[a]["bleu"] for a in ORDER]
ys = [ARMS[a]["comet"] for a in ORDER]
ax.scatter(xs, ys, s=[70 if ARMS[a]["g"] in ("ce", "kl") else 45 for a in ORDER],
           color=[COL[ARMS[a]["g"]] for a in ORDER], zorder=3, edgecolor="w", linewidth=0.8)
for a in ["ridge_full", "ce_ridge_e6", "ce_ridge_e16", "ce_scratch"]:
    ax.annotate(ARMS[a]["short"], (ARMS[a]["bleu"], ARMS[a]["comet"]), xytext=(6, -10),
                textcoords="offset points", fontsize=7.6, color=MUT)
ax.set_xlabel("BLEU"); ax.set_ylabel("COMET")
ax.set_title("(b)  an independent metric agrees", loc="left")
ax.text(0.04, 0.95, "Spearman $\\rho = 1.0$\nacross 9 arms", transform=ax.transAxes,
        fontsize=8.6, va="top")

fig.suptitle("Not a split artefact, and not a BLEU artefact", fontsize=11.2, x=0.02, ha="left")
save(fig, "fig12_replication")

wrote figures\fig12_replication.png (np.float64(10.2), np.float64(3.9))


<Figure size 1122x429 with 2 Axes>

## 13. Numbers used in the paper's tables

A single printout so the tables can be checked against the notebook without opening the JSON.

In [14]:
rows = []
for a in ORDER:
    d = ARMS[a]
    rows.append(dict(arm=a, bleu=d["bleu"], bleu48=M48["arms"][a]["bleu"], chrf=d["chrf"],
                     ce=d["ce"], degen=d["degen"], comet=d["comet"], norm=d["norm"], zdist=d["zdist"]))
import pandas as pd
t = pd.DataFrame(rows).set_index("arm")
t.loc["native"] = [NATIVE["bleu"], M48["arms"].get("native", {}).get("bleu", np.nan),
                   NATIVE["chrf"], NATIVE["ce"], NATIVE["degen"], np.nan, NATIVE["norm"], np.nan]
t.loc["oracle"] = [ORACLE["bleu"], np.nan, ORACLE["chrf"], ORACLE["ce"], ORACLE["degen"],
                   np.nan, ORACLE["norm"], np.nan]
t.round(4)

,bleu,bleu48,chrf,ce,degen,comet,norm,zdist
arm,,,,,,,,
ce_scratch,2.245,2.433,17.280,3.3140,0.265,0.4363,24.40,33.7
ridge_sub,2.474,2.667,19.680,3.2320,0.167,0.4751,7.60,6.2
ridge_full,2.499,2.711,19.770,3.2280,0.155,0.4797,7.60,6.2
ridge_scale122,2.888,3.212,20.880,3.2480,0.190,0.4898,9.30,7.9
ridge_recolor10,3.085,3.431,20.700,3.2620,0.144,0.4966,8.80,8.6
ce_ridge,5.332,5.725,25.000,2.8570,0.118,0.5331,9.30,10.6
ce_ridge_e6,7.045,7.593,27.240,2.7610,0.090,0.5641,11.30,14.3
distill,7.254,7.767,27.570,2.7410,0.104,0.5678,11.90,15.4
ce_ridge_e16,7.897,8.456,29.360,2.7380,0.067,0.5830,13.80,18.8
